# Fire Season Timing Graphs - Turkey Ecoregion Scale 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns
from scipy import stats

# Load master CSV
output_dir = 'outputs/turkey_ecoregions'
master_df  = pd.read_csv(f'{output_dir}/master_turkey.csv')

print(f'Loaded {len(master_df)} rows')
print(f'Ecoregions: {master_df["eco_name"].nunique()}')
print(f'Years: {master_df["year"].min()} – {master_df["year"].max()}')
print()
print(master_df.head())

In [ ]:
graphs_dir = 'outputs/turkey_ecoregions/graphs'
os.makedirs(graphs_dir, exist_ok=True)

In [ ]:
n_ecos    = master_df['eco_name'].nunique()
eco_names = master_df['eco_name'].unique()

fig, axes = plt.subplots(n_ecos, 4, figsize=(20, n_ecos * 2.5))
fig.suptitle('Fire Season Timing — Turkey Ecoregions (2003–2024)',
             fontsize=14, y=1.01)

metrics = [
    ('onset_doy',     'Onset DOY',            'orange'),
    ('peak_doy',      'Peak DOY (centroid)',   'firebrick'),
    ('end_doy',       'End DOY',              'steelblue'),
    ('season_length', 'Season Length (days)', 'black'),
]

for row_idx, eco_name in enumerate(sorted(eco_names)):
    eco_df = master_df[master_df['eco_name'] == eco_name].sort_values('year')

    for col_idx, (metric, title, color) in enumerate(metrics):
        ax = axes[row_idx, col_idx]
        ax.plot(eco_df['year'], eco_df[metric],
                marker='o', color=color, linewidth=1.5, markersize=4)
        ax.set_ylim(1, 366)      # same for all panels
        ax.grid(True, alpha=0.3)
        ax.set_xticks(eco_df['year'])
        ax.tick_params(axis='x', rotation=90, labelsize=7)

        if row_idx == 0:
            ax.set_title(title, fontsize=10)
        if col_idx == 0:
            ax.set_ylabel(eco_name.replace(' ', '\n'), fontsize=10, rotation=0, labelpad=10)
            ax.yaxis.label.set_ha('right')
            ax.yaxis.label.set_va('center')

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_01_timing_per_ecoregion.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle('Fire Season Timing Heatmaps — Turkey Ecoregions (2003–2024)', fontsize=13)

metrics = [
    ('onset_doy',     'Onset DOY',            'RdYlBu_r'),  # blue=early, red=late
    ('peak_doy',      'Peak DOY (centroid)',   'RdYlBu_r'),
    ('end_doy',       'End DOY',              'RdYlBu_r'),
    ('season_length', 'Season Length (days)', 'pink_r'),    # red=short, green=long
]

for ax, (metric, title, cmap) in zip(axes.flatten(), metrics):
    pivot = master_df.pivot(index='eco_name', columns='year', values=metric)

    # Diverging: center the colormap around the median value of each metric
    vmin   = pivot.values.min()
    vmax   = pivot.values.max()
    # Only center diverging colormaps (onset, peak, end) — not season_length
    center = np.nanmean(pivot.values) if metric != 'season_length' else None
    
    sns.heatmap(
        pivot,
        ax         = ax,
        cmap       = cmap,
        center     = center,
        vmin       = vmin,
        vmax       = vmax,
        linewidths = 0.4,
        linecolor  = 'white',
        annot      = False,
        cbar_kws   = {'shrink': 0.8, 'label': metric}
    )

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=8)
    ax.tick_params(axis='x', labelsize=8, rotation=90)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_02_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Fire Season Timing Trends — Turkey Ecoregions (2003–2024)', fontsize=13)

metrics = [
    ('onset_doy',     'Onset DOY',            'orange'),
    ('peak_doy',      'Peak DOY (centroid)',   'firebrick'),
    ('end_doy',       'End DOY',              'steelblue'),
    ('season_length', 'Season Length (days)', 'green'),
]

for ax, (metric, title, color) in zip(axes.flatten(), metrics):
    # Plot each ecoregion as a faint line
    for eco_name, group in master_df.groupby('eco_name'):
        ax.plot(group['year'], group[metric],
                color=color, alpha=0.2, linewidth=1)

    # Plot the cross-ecoregion mean per year as a bold line
    yearly_mean = master_df.groupby('year')[metric].mean()
    yearly_std  = master_df.groupby('year')[metric].std()
    years       = yearly_mean.index

    ax.plot(years, yearly_mean, color=color, linewidth=2.5, label='Mean')
    ax.fill_between(years,
                    yearly_mean - yearly_std,
                    yearly_mean + yearly_std,
                    color=color, alpha=0.15, label='±1 SD')

    # Fit and plot overall trend line
    slope, intercept, r, p, se = stats.linregress(years, yearly_mean)
    trend = np.array([slope * y + intercept for y in years])

    # 95% confidence interval around the trend line
    # se is the standard error of the slope
    n        = len(years)
    t_crit   = stats.t.ppf(0.975, df=n - 2)  # 95% two-tailed critical value
    x_mean   = np.mean(years)
    ss_x     = np.sum((np.array(years) - x_mean) ** 2)
    ci       = t_crit * se * np.sqrt(1/n + (np.array(years) - x_mean)**2 / ss_x)

    ax.plot(years, trend, color='black', linewidth=1.5,
        linestyle='--', label=f'Trend (p={p:.2f}, slope={slope:.2f}/yr)')
    ax.fill_between(years, trend - ci, trend + ci,
                color='black', alpha=0.1, label='95% CI')

    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Year')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(years)
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_03_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import ee
import geopandas as gpd
from shapely.geometry import shape

# Initialize GEE
ee.Authenticate()
ee.Initialize(project='fire-seasons')

# Re-fetch Turkey ecoregions
turkey = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
           .filter(ee.Filter.eq('country_na', 'Turkey'))

ecoregions_turkey = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017") \
                      .filterBounds(turkey.geometry())

eco_list = ecoregions_turkey.select(['ECO_ID', 'ECO_NAME']).getInfo()

print(f'Loaded {len(eco_list["features"])} ecoregions.')

In [ ]:
import geopandas as gpd
from shapely.geometry import shape

# Compute mean metrics per ecoregion across all years
mean_metrics = master_df.groupby('eco_name')[
    ['onset_doy', 'peak_doy', 'end_doy', 'season_length']
].mean().reset_index()

# Build GeoDataFrame directly from the GEE features we already loaded
# eco_list was fetched earlier — it has both geometries and properties
rows = []
for f in eco_list['features']:
    eco_name = f['properties']['ECO_NAME']
    if eco_name in mean_metrics['eco_name'].values:
        rows.append({
            'eco_name': eco_name,
            'geometry': shape(f['geometry'])
        })

eco_gdf    = gpd.GeoDataFrame(rows, crs='EPSG:4326')
eco_merged = eco_gdf.merge(mean_metrics, on='eco_name')

# Plot
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Mean Fire Season Timing — Turkey Ecoregions', fontsize=13)

metrics = [
    ('onset_doy',     'Mean Onset DOY',           'YlOrRd'),
    ('peak_doy',      'Mean Peak DOY (centroid)',  'YlOrRd'),
    ('end_doy',       'Mean End DOY',              'YlOrRd'),
    ('season_length', 'Mean Season Length (days)', 'YlGn'),
]

for ax, (metric, title, cmap) in zip(axes.flatten(), metrics):
    eco_merged.plot(column=metric, ax=ax, cmap=cmap,
                    legend=True, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=10)
    ax.set_axis_off()
plt.show()
plt.savefig(f'{graphs_dir}/graph_04_map.png', dpi=150, bbox_inches='tight')


In [ ]:
from pandas.plotting import scatter_matrix

cols = ['onset_doy', 'peak_doy', 'end_doy', 'season_length']

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Metric Relationships — Turkey Ecoregions (2003–2024)', fontsize=13)

scatter_matrix(
    master_df[cols],
    ax           = axes,
    alpha        = 0.3,
    diagonal     = 'hist',
    color        = 'steelblue',
    hist_kwds    = {'bins': 15, 'color': 'steelblue', 'alpha': 0.7}
)

# Add correlation values to each panel
for i, col1 in enumerate(cols):
    for j, col2 in enumerate(cols):
        if i != j:
            r, p = stats.pearsonr(
                master_df[col1].dropna(),
                master_df[col2].dropna()
            )
            axes[i, j].annotate(f'r={r:.2f}', xy=(0.05, 0.88),
                                 xycoords='axes fraction', fontsize=8,
                                 color='firebrick')

plt.tight_layout()
plt.savefig(f'{graphs_dir}/graph_05_scatter_matrix.png', dpi=150, bbox_inches='tight')
plt.show()